# Find one match and check its closing odds

An analyst needs Bet365's closing home, draw and away prices for one match. The same teams can meet more than once. Match the division, date and both teams before reading prices.

Run all cells for an offline exercise with invented data. To try a real season, change the setting below to `"publisher"` and run all cells again. That makes one request for [Football-Data.co.uk's 2024/25 English Premier League CSV](https://www.football-data.co.uk/mmz4281/2425/E0.csv). No ParlayAPI account or key is needed.

In [ ]:
SOURCE = "practice"  # "practice" or "publisher"
# Division, date as written in the CSV, home team, away team
PRACTICE_MATCH = ("T0", "16/08/2024", "Harbor FC", "Valley FC")
PUBLISHER_MATCH = ("E0", "16/08/2024", "Man United", "Fulham")

## 1. Identify the columns

`Div`, `Date`, `HomeTeam` and `AwayTeam` identify the requested fixture in this file. `B365CH`, `B365CD` and `B365CA` hold the closing decimal prices for a home win, draw and away win. The corresponding `B365H`, `B365D` and `B365A` columns hold pre-closing odds. Keep the two observations separate. [Publisher's column definitions](https://www.football-data.co.uk/notes.txt)

In [ ]:
import csv
import io
from decimal import Decimal, InvalidOperation
from urllib.request import Request, urlopen

PUBLISHER_CSV = "https://www.football-data.co.uk/mmz4281/2425/E0.csv"
IDENTITY = ("Div", "Date", "HomeTeam", "AwayTeam")
CLOSING = {"home": "B365CH", "draw": "B365CD", "away": "B365CA"}
PRACTICE_CSV = """Div,Date,HomeTeam,AwayTeam,B365H,B365D,B365A,B365CH,B365CD,B365CA
T0,16/08/2024,Harbor FC,Valley FC,2.05,3.20,3.90,2.10,,3.95
T0,23/08/2024,Harbor FC,Valley FC,2.15,3.40,3.50,2.20,3.45,3.55
T0,16/08/2024,Valley FC,Harbor FC,2.50,3.30,2.90,2.60,3.35,2.95
"""

if SOURCE == "practice":
    csv_text, target, source_label = PRACTICE_CSV, PRACTICE_MATCH, "Invented coursework rows"
elif SOURCE == "publisher":
    request = Request(PUBLISHER_CSV, headers={"User-Agent": "ParlayAPI historical match lesson"})
    with urlopen(request, timeout=20) as response:
        csv_text = response.read().decode("utf-8-sig")
    target, source_label = PUBLISHER_MATCH, PUBLISHER_CSV
else:
    raise ValueError('Set SOURCE to "practice" or "publisher".')

print("Source:", source_label)
print("Requested fixture:", target)

## 2. Resolve the fixture before inspecting its prices

A missing fixture and a fixture with a missing draw price answer different questions. If two rows match the full identity, stop and inspect the duplicates. The function never borrows a price from another row or from a pre-closing column.

In [ ]:
def closing_report(csv_text, target):
    reader = csv.DictReader(io.StringIO(csv_text))
    fields = reader.fieldnames or []
    required = set(IDENTITY) | set(CLOSING.values())
    if len(fields) != len(set(fields)) or not required.issubset(fields):
        raise ValueError("The input needs unique fixture columns and all three B365C closing columns.")
    matches = [row for row in reader if tuple(row[key] for key in IDENTITY) == target]
    report = {"fixture": dict(zip(IDENTITY, target)), "matching_rows": len(matches)}
    if len(matches) != 1:
        report["status"] = "no_match" if not matches else "ambiguous"
        return report

    prices, issues = {}, {}
    for outcome, column in CLOSING.items():
        raw = (matches[0][column] or "").strip()
        try:
            price = Decimal(raw)
            valid = price.is_finite() and price > 1
        except InvalidOperation:
            valid = False
        prices[outcome] = raw if valid else None
        if not valid:
            issues[outcome] = "missing" if not raw else "invalid"
    report.update(status="incomplete" if issues else "complete", closing_decimal=prices, issues=issues)
    return report

report = closing_report(csv_text, target)
print(report)

## 3. Try the exercise

1. In practice mode, identify the outcome missing for Harbor FC at home on 16 August. Can you defend a complete three-way comparison?
2. Change the practice date to 23 August. Which part of the fixture identity changed? Are all three closing outcomes available?
3. Switch to publisher mode and restore the original settings for the real example. Use the source link to check the selected row and closing columns yourself. The file supplies a historical closing price, but the match date gives no exact quote timestamp.

**Answer key for the invented rows.** On 16 August the closing draw is missing. The pre-closing draw of 3.20 and the 23 August closing draw of 3.45 each describe a different observation. On 23 August the three closing prices are 2.20, 3.45 and 3.55. Reversing home and away also identifies a different fixture.

Data for publisher mode comes from [Football-Data.co.uk](https://www.football-data.co.uk/englandm.php). ParlayAPI maintains this lesson. For a trial using your own API response, use the [evaluation lab](https://parlay-api.com/tools/evaluation-lab) to record the event, bookmaker, market and fields you need.